# Results and Analysis

This notebook consolidates the executed result tables for the Arabic deepfake detection study across Twitter and YouTube. It covers in-domain performance, preprocessing effects, error analysis, cross-platform transfer, and the final hypothesis decisions.

All tables below are derived from the exported CSV artifacts produced by the executed pipelines rather than from hand-entered values.

## Reading Guide and Statistical Method

This notebook is organized in the same order as the paper results section.

- `Macro-F1` remains the primary metric.
- `Exact McNemar` tests are used when two systems are evaluated on the same held-out items.
- `Wilcoxon signed-rank` tests are used for family-wise preprocessing summaries over matched configurations.
- `95% bootstrap confidence intervals` are used for reported Macro-F1 uncertainty and paired differences where available.
- `BH-FDR` correction is applied across grouped p-value families in the reported tables.

Important interpretation note: p-values are used to test null hypotheses of equal performance. The notebook therefore reports whether the evidence supports each research hypothesis, not whether a hypothesis itself was literally rejected.

Analysis generated on 2026-04-06.

## In-Domain Performance Overview

### Exact-model overview by platform and preprocessing view

| platform   | preprocessing   |   ml_mean |   sequence_mean |   transformer_mean | best_model   |   best_macro_f1 | worst_model   |   worst_macro_f1 |   spread_best_minus_worst |
|:-----------|:----------------|----------:|----------------:|-------------------:|:-------------|----------------:|:--------------|-----------------:|--------------------------:|
| twitter    | deepfake_aware  |    0.9548 |          0.9736 |             0.9857 | marbertv2    |          0.9869 | randomforest  |           0.9324 |                    0.0545 |
| twitter    | manual          |    0.9093 |          0.9344 |             0.9596 | marbertv2    |          0.9625 | randomforest  |           0.877  |                    0.0855 |
| twitter    | original        |    0.9581 |          0.9757 |             0.9906 | camelbert    |          0.9912 | randomforest  |           0.9135 |                    0.0777 |
| youtube    | deepfake_aware  |    0.8645 |          0.8888 |             0.9318 | marbertv2    |          0.9369 | randomforest  |           0.825  |                    0.1119 |
| youtube    | manual          |    0.8587 |          0.8826 |             0.9303 | marbertv2    |          0.9354 | randomforest  |           0.8225 |                    0.113  |
| youtube    | original        |    0.8678 |          0.8898 |             0.939  | marbertv2    |          0.9409 | randomforest  |           0.8297 |                    0.1112 |

### Best configuration within each family and preprocessing view

| platform   | preprocessing   | ml_model           |   ml_macro_f1 |   ml_delta_vs_original | sequence_model   |   sequence_macro_f1 |   sequence_delta_vs_original | transformer_model   |   transformer_macro_f1 |   transformer_delta_vs_original |
|:-----------|:----------------|:-------------------|--------------:|-----------------------:|:-----------------|--------------------:|-----------------------------:|:--------------------|-----------------------:|--------------------------------:|
| twitter    | deepfake_aware  | linearsvc          |        0.971  |                -0.0014 | lstm             |              0.977  |                      -0.0042 | marbertv2           |                 0.9869 |                         -0.0042 |
| twitter    | manual          | linearsvc          |        0.9264 |                -0.046  | bilstm           |              0.9445 |                      -0.0368 | marbertv2           |                 0.9625 |                         -0.0286 |
| twitter    | original        | linearsvc          |        0.9724 |                 0      | bilstm           |              0.9813 |                       0      | camelbert           |                 0.9912 |                          0      |
| youtube    | deepfake_aware  | linearsvc          |        0.8905 |                -0.0046 | bilstm           |              0.8973 |                       0.0003 | marbertv2           |                 0.9369 |                         -0.0041 |
| youtube    | manual          | logisticregression |        0.8822 |                -0.0129 | bilstm           |              0.8977 |                       0.0006 | marbertv2           |                 0.9354 |                         -0.0055 |
| youtube    | original        | logisticregression |        0.8951 |                 0      | bilstm           |              0.8971 |                       0      | marbertv2           |                 0.9409 |                          0      |

**Interpretation.** Across all six platform-view conditions, the transformer mean Macro-F1 is the highest family mean. The strongest in-domain result appears on Twitter under the original view (camelbert, Macro-F1=0.9912), while the strongest YouTube result is also under the original view (marbertv2, Macro-F1=0.9409). Manual preprocessing produces the clearest drop in family means on Twitter, especially for the sequence and transformer families, and YouTube shows the same direction of degradation for the transformer family even though the absolute drop is smaller than on Twitter. The larger spreads on YouTube (0.1112 under the original view versus 0.0777 on Twitter) also show that configuration choice matters more on YouTube.

## H1: Fine-tuned contextual Arabic transformers will produce the strongest in-domain Macro-F1 on both platforms

### Family-level test used to decide H1

| platform   | comparison_family   | transformer_model   | transformer_preprocessing   | comparison_model   | comparison_preprocessing   |   macro_f1_delta |   bootstrap_ci_lower |   bootstrap_ci_upper |   p_value |   p_adjusted | significance_flag   | decision   |
|:-----------|:--------------------|:--------------------|:----------------------------|:-------------------|:---------------------------|-----------------:|---------------------:|---------------------:|----------:|-------------:|:--------------------|:-----------|
| twitter    | ML                  | camelbert           | original                    | linearsvc          | original                   |           0.0187 |               0.0131 |               0.0247 |  1.55e-10 |     4.66e-10 | True                | Support H1 |
| twitter    | Sequence            | camelbert           | original                    | bilstm             | original                   |           0.0099 |               0.0046 |               0.0152 |  0.0003   |     0.0005   | True                | Support H1 |
| youtube    | ML                  | marbertv2           | original                    | logisticregression | original                   |           0.0458 |               0.0375 |               0.0539 |  1.26e-27 |     3.78e-27 | True                | Support H1 |
| youtube    | Sequence            | marbertv2           | original                    | bilstm             | manual                     |           0.0433 |               0.0352 |               0.0512 |  2.47e-25 |     3.7e-25  | True                | Support H1 |

### Transformer-only view summary

| platform   | preprocessing   | best_transformer   |   best_macro_f1 | runner_up_transformer   |   runner_up_macro_f1 |   delta_best_minus_runner_up |   p_value |   p_adjusted | runner_up_significant_after_bh   |
|:-----------|:----------------|:-------------------|----------------:|:------------------------|---------------------:|-----------------------------:|----------:|-------------:|:---------------------------------|
| twitter    | deepfake_aware  | marbertv2          |          0.9869 | arabertv2               |               0.9855 |                       0.0014 |    0.5572 |       0.6728 | False                            |
| twitter    | manual          | marbertv2          |          0.9625 | camelbert               |               0.959  |                       0.0035 |    0.3143 |       0.3305 | False                            |
| twitter    | original        | camelbert          |          0.9912 | marbertv2               |               0.9908 |                       0.0004 |    1      |       1      | False                            |
| youtube    | deepfake_aware  | marbertv2          |          0.9369 | arabertv2               |               0.9296 |                       0.0072 |    0.0201 |       0.0252 | True                             |
| youtube    | manual          | marbertv2          |          0.9354 | camelbert               |               0.928  |                       0.0074 |    0.0115 |       0.0144 | True                             |
| youtube    | original        | marbertv2          |          0.9409 | arabertv2               |               0.94   |                       0.0009 |    0.8147 |       0.8147 | False                            |

### Transformer-only pairwise view comparisons

| platform   | preprocessing   | best_transformer   |   best_macro_f1 | competitor_transformer   |   competitor_macro_f1 |   delta_best_minus_competitor |   p_value |   p_adjusted | significant_after_bh   |
|:-----------|:----------------|:-------------------|----------------:|:-------------------------|----------------------:|------------------------------:|----------:|-------------:|:-----------------------|
| twitter    | deepfake_aware  | marbertv2          |          0.9869 | arabertv2                |                0.9855 |                        0.0014 |    0.5572 |       0.6728 | False                  |
| twitter    | deepfake_aware  | marbertv2          |          0.9869 | camelbert                |                0.9848 |                        0.0021 |    0.3616 |       0.4931 | False                  |
| twitter    | manual          | marbertv2          |          0.9625 | arabertv2                |                0.9572 |                        0.0053 |    0.1462 |       0.1827 | False                  |
| twitter    | manual          | marbertv2          |          0.9625 | camelbert                |                0.959  |                        0.0035 |    0.3143 |       0.3305 | False                  |
| twitter    | original        | camelbert          |          0.9912 | arabertv2                |                0.9897 |                        0.0014 |    0.5572 |       0.6429 | False                  |
| twitter    | original        | camelbert          |          0.9912 | marbertv2                |                0.9908 |                        0.0004 |    1      |       1      | False                  |
| youtube    | deepfake_aware  | marbertv2          |          0.9369 | arabertv2                |                0.9296 |                        0.0072 |    0.0201 |       0.0252 | True                   |
| youtube    | deepfake_aware  | marbertv2          |          0.9369 | camelbert                |                0.9291 |                        0.0078 |    0.0141 |       0.0193 | True                   |
| youtube    | manual          | marbertv2          |          0.9354 | arabertv2                |                0.9273 |                        0.0081 |    0.0075 |       0.0102 | True                   |
| youtube    | manual          | marbertv2          |          0.9354 | camelbert                |                0.928  |                        0.0074 |    0.0115 |       0.0144 | True                   |
| youtube    | original        | marbertv2          |          0.9409 | arabertv2                |                0.94   |                        0.0009 |    0.8147 |       0.8147 | False                  |
| youtube    | original        | marbertv2          |          0.9409 | camelbert                |                0.9362 |                        0.0048 |    0.1019 |       0.1274 | False                  |

**Interpretation.** This H1 table compares the best overall in-domain transformer setting on each platform against the best overall sequence and ML settings on that same platform, where family winners are selected by Macro-F1 and then accuracy. On Twitter, the best transformer outperformed the best sequence model (adjusted p=0.0005, delta=0.0099) and the best ML model (adjusted p=4.66e-10, delta=0.0187). On YouTube, the best transformer also outperformed the best sequence model (adjusted p=3.70e-25, delta=0.0433) and the best ML model (adjusted p=3.78e-27, delta=0.0458). Within the transformer family, significant within-view backbone differences appeared only in youtube-deepfake_aware (marbertv2, adjusted p=0.0252), youtube-manual (marbertv2, adjusted p=0.0144). This supports H1 at the platform level using the strongest available setting from each family.

## Preprocessing Effects and H2

The preprocessing analysis is reported in two layers. The first layer uses family-wise Wilcoxon tests on matched configuration deltas. The second layer uses paired prediction-level McNemar tests for the most important best-configuration comparisons against the original view.

### Family-wise preprocessing tests already exported by the pipelines

| platform   | model_family   | reference_preprocessing   | variant_preprocessing   |   mean_macro_f1_delta |   median_macro_f1_delta |   p_value |   p_adjusted |   rank_biserial_correlation |   n_pairs | favors   | significance_flag   |
|:-----------|:---------------|:--------------------------|:------------------------|----------------------:|------------------------:|----------:|-------------:|----------------------------:|----------:|:---------|:--------------------|
| twitter    | ML             | original                  | deepfake_aware          |               -0.0033 |                 -0.0078 |    0.4258 |       0.4258 |                     -0.3333 |         9 | original | False               |
| twitter    | ML             | original                  | manual                  |               -0.0487 |                 -0.0489 |    0.0039 |       0.0234 |                     -1      |         9 | original | True                |
| twitter    | Sequence       | original                  | deepfake_aware          |               -0.0021 |                 -0.0016 |    0.3125 |       0.375  |                     -0.5238 |         6 | original | False               |
| twitter    | Sequence       | original                  | manual                  |               -0.0413 |                 -0.0414 |    0.0312 |       0.0938 |                     -1      |         6 | original | False               |
| twitter    | Transformer    | original                  | deepfake_aware          |               -0.0048 |                 -0.0042 |    0.25   |       0.375  |                     -1      |         3 | original | False               |
| twitter    | Transformer    | original                  | manual                  |               -0.031  |                 -0.0322 |    0.25   |       0.375  |                     -1      |         3 | original | False               |
| youtube    | ML             | original                  | deepfake_aware          |               -0.0033 |                 -0.0035 |    0.0117 |       0.0352 |                     -0.9111 |         9 | original | True                |
| youtube    | ML             | original                  | manual                  |               -0.0091 |                 -0.0074 |    0.0039 |       0.0234 |                     -1      |         9 | original | True                |
| youtube    | Sequence       | original                  | deepfake_aware          |               -0.001  |                 -0      |    0.8438 |       0.8438 |                     -0.1429 |         6 | original | False               |
| youtube    | Sequence       | original                  | manual                  |               -0.0072 |                 -0.0078 |    0.0625 |       0.125  |                     -0.9048 |         6 | original | False               |
| youtube    | Transformer    | original                  | deepfake_aware          |               -0.0072 |                 -0.0071 |    0.25   |       0.3    |                     -1      |         3 | original | False               |
| youtube    | Transformer    | original                  | manual                  |               -0.0088 |                 -0.0081 |    0.25   |       0.3    |                     -1      |         3 | original | False               |

### Best-config preprocessing comparisons for the most important practical contrasts

| platform   | model_family   | comparison                 | reference_preprocessing   | variant_preprocessing   | original_model     | original_config              | variant_model      | variant_config               |   macro_f1_original |   macro_f1_variant |   macro_f1_delta_variant_minus_original |   bootstrap_ci_lower |   bootstrap_ci_upper |   discordant_variant_only |   discordant_original_only |   p_value |   n_samples | favors         |   p_adjusted | significance_flag   |
|:-----------|:---------------|:---------------------------|:--------------------------|:------------------------|:-------------------|:-----------------------------|:-------------------|:-----------------------------|--------------------:|-------------------:|----------------------------------------:|---------------------:|---------------------:|--------------------------:|---------------------------:|----------:|------------:|:---------------|-------------:|:--------------------|
| twitter    | ML             | original_vs_deepfake_aware | original                  | deepfake_aware          | linearsvc          | linearsvc__fasttext          | linearsvc          | linearsvc__tfidf             |              0.9724 |             0.971  |                                 -0.0014 |              -0.0088 |               0.0053 |                        47 |                         51 |  0.762    |        2829 | original       |     0.9144   | False               |
| twitter    | ML             | original_vs_manual         | original                  | manual                  | linearsvc          | linearsvc__fasttext          | linearsvc          | linearsvc__tfidf             |              0.9724 |             0.9264 |                                 -0.046  |              -0.0562 |              -0.0364 |                        37 |                        167 |  6.38e-21 |        2829 | original       |     7.65e-20 | True                |
| twitter    | Sequence       | original_vs_deepfake_aware | original                  | deepfake_aware          | bilstm             | bilstm__fasttext             | lstm               | lstm__fasttext               |              0.9813 |             0.977  |                                 -0.0042 |              -0.0106 |               0.0018 |                        31 |                         43 |  0.2007   |        2829 | original       |     0.2676   | False               |
| twitter    | Sequence       | original_vs_manual         | original                  | manual                  | bilstm             | bilstm__fasttext             | bilstm             | bilstm__fasttext             |              0.9813 |             0.9445 |                                 -0.0368 |              -0.0449 |              -0.0291 |                        17 |                        121 |  1.6e-20  |        2829 | original       |     9.6e-20  | True                |
| twitter    | Transformer    | original_vs_deepfake_aware | original                  | deepfake_aware          | camelbert          | camelbert                    | marbertv2          | marbertv2                    |              0.9912 |             0.9869 |                                 -0.0042 |              -0.0078 |              -0.0007 |                         7 |                         19 |  0.029    |        2829 | original       |     0.0695   | False               |
| twitter    | Transformer    | original_vs_manual         | original                  | manual                  | camelbert          | camelbert                    | marbertv2          | marbertv2                    |              0.9912 |             0.9625 |                                 -0.0286 |              -0.0357 |              -0.0219 |                         9 |                         90 |  6.05e-18 |        2829 | original       |     2.42e-17 | True                |
| youtube    | ML             | original_vs_deepfake_aware | original                  | deepfake_aware          | logisticregression | logisticregression__fasttext | linearsvc          | linearsvc__fasttext          |              0.8951 |             0.8905 |                                 -0.0046 |              -0.0103 |               0.0011 |                       113 |                        139 |  0.1151   |        5654 | original       |     0.1727   | False               |
| youtube    | ML             | original_vs_manual         | original                  | manual                  | logisticregression | logisticregression__fasttext | logisticregression | logisticregression__fasttext |              0.8951 |             0.8822 |                                 -0.0129 |              -0.0188 |              -0.0067 |                       114 |                        187 |  3.06e-05 |        5654 | original       |     9.18e-05 | True                |
| youtube    | Sequence       | original_vs_deepfake_aware | original                  | deepfake_aware          | bilstm             | bilstm__fasttext             | bilstm             | bilstm__fasttext             |              0.8971 |             0.8973 |                                  0.0003 |              -0.0066 |               0.007  |                       188 |                        185 |  0.9175   |        5654 | deepfake_aware |     0.9175   | False               |
| youtube    | Sequence       | original_vs_manual         | original                  | manual                  | bilstm             | bilstm__fasttext             | bilstm             | bilstm__fasttext             |              0.8971 |             0.8977 |                                  0.0006 |              -0.0055 |               0.0068 |                       161 |                        157 |  0.8664   |        5654 | manual         |     0.9175   | False               |
| youtube    | Transformer    | original_vs_deepfake_aware | original                  | deepfake_aware          | marbertv2          | marbertv2                    | marbertv2          | marbertv2                    |              0.9409 |             0.9369 |                                 -0.0041 |              -0.0084 |               0.0003 |                        67 |                         90 |  0.0788   |        5654 | original       |     0.1351   | False               |
| youtube    | Transformer    | original_vs_manual         | original                  | manual                  | marbertv2          | marbertv2                    | marbertv2          | marbertv2                    |              0.9409 |             0.9354 |                                 -0.0055 |              -0.0106 |              -0.0005 |                        86 |                        117 |  0.035    |        5654 | original       |     0.07     | False               |

**Interpretation.** The strict version of H2 is not fully supported. Family-wise Wilcoxon tests show significant preprocessing effects only for selected contrasts rather than uniformly across all families: twitter-ML-manual (adjusted p=0.0234), youtube-ML-deepfake_aware (adjusted p=0.0352), youtube-ML-manual (adjusted p=0.0234). The best-config paired McNemar analysis tells a similar story, with significant effects concentrated in twitter-ML-manual (adjusted p=7.65e-20), twitter-Sequence-manual (adjusted p=9.60e-20), twitter-Transformer-manual (adjusted p=2.42e-17), youtube-ML-manual (adjusted p=9.18e-05). In practical terms, manual preprocessing is the clearest source of degradation, especially on Twitter, whereas deepfake-aware preprocessing is usually smaller and often not significant after multiple-comparison correction. The safest wording is therefore that preprocessing effects are real but selective and non-uniform.

## Error Analysis

### Mean exact-model error rates by deceptive type

| deceptive_type     |   twitter_mean_error_pct |   youtube_mean_error_pct |
|:-------------------|-------------------------:|-------------------------:|
| original           |                   4.2822 |                  10.0145 |
| clickbait phrasing |                   0.5359 |                   3.8264 |
| contradiction      |                   8.1634 |                  24.686  |
| exaggeration       |                   1.5826 |                   7.0981 |
| mixed truths       |                   1.8519 |                  13.198  |
| omission           |                  17.6737 |                  24.422  |
| satirical tone     |                   1.3391 |                   7.0192 |

**Interpretation.** Error patterns are not uniform across deception types. On Twitter, the hardest category is `omission` with a mean error rate of 17.7%, far above the next cluster of error types. On YouTube, `contradiction` (24.7%) and `omission` (24.4%) dominate the failure profile. This indicates that the hardest cases are not the most visibly sensational texts, but those that omit key information, reverse meaning, or mix true and false content while remaining fluent.

## Cross-Platform Evaluation and H3

### Best transfer-performing family configuration by preprocessing view

| source_platform   | target_platform   | preprocessing   | ml_model           |   ml_macro_f1 | sequence_model   |   sequence_macro_f1 | transformer_model   |   transformer_macro_f1 |
|:------------------|:------------------|:----------------|:-------------------|--------------:|:-----------------|--------------------:|:--------------------|-----------------------:|
| twitter           | youtube           | deepfake_aware  | linearsvc          |        0.7613 | bilstm           |              0.6506 | camelbert           |                 0.7206 |
| twitter           | youtube           | manual          | logisticregression |        0.7861 | bilstm           |              0.8048 | marbertv2           |                 0.8364 |
| twitter           | youtube           | original        | linearsvc          |        0.6904 | bilstm           |              0.6569 | camelbert           |                 0.7461 |
| youtube           | twitter           | deepfake_aware  | linearsvc          |        0.7552 | lstm             |              0.703  | arabertv2           |                 0.9099 |
| youtube           | twitter           | manual          | randomforest       |        0.77   | bilstm           |              0.7129 | arabertv2           |                 0.907  |
| youtube           | twitter           | original        | linearsvc          |        0.7453 | bilstm           |              0.7358 | arabertv2           |                 0.9433 |

### Family-level transfer-loss tests

| source_platform   | target_platform   | model_family   |   mean_macro_f1_delta |   median_macro_f1_delta |   p_value |   p_adjusted |   rank_biserial_correlation | significance_flag   |
|:------------------|:------------------|:---------------|----------------------:|------------------------:|----------:|-------------:|----------------------------:|:--------------------|
| twitter           | youtube           | ML             |                0.2424 |                  0.2195 |  1.49e-08 |     2.24e-07 |                      1      | True                |
| twitter           | youtube           | Sequence       |                0.2844 |                  0.3268 |  7.63e-06 |     2.29e-05 |                      1      | True                |
| twitter           | youtube           | Transformer    |                0.2332 |                  0.2541 |  0.0039   |     0.0065   |                      1      | True                |
| youtube           | twitter           | ML             |                0.1938 |                  0.1896 |  1.49e-08 |     2.24e-07 |                      1      | True                |
| youtube           | twitter           | Sequence       |                0.2737 |                  0.2577 |  7.63e-06 |     2.86e-05 |                      1      | True                |
| youtube           | twitter           | Transformer    |                0.0358 |                  0.0274 |  0.0078   |     0.013    |                      0.9556 | True                |

### Bootstrap confidence intervals for the strongest transfer configuration in each family

| source_platform   | target_platform   | model_family   | model_name         | config_name               | preprocessing   |   macro_f1_delta |   ci_lower |   ci_upper | ci_excludes_zero   |
|:------------------|:------------------|:---------------|:-------------------|:--------------------------|:----------------|-----------------:|-----------:|-----------:|:-------------------|
| twitter           | youtube           | ML             | logisticregression | logisticregression__tfidf | manual          |           0.1346 |     0.1206 |     0.1489 | True               |
| twitter           | youtube           | Sequence       | bilstm             | bilstm__word2vec_cbow     | manual          |           0.1322 |     0.1189 |     0.1465 | True               |
| twitter           | youtube           | Transformer    | marbertv2          | marbertv2                 | manual          |           0.1261 |     0.114  |     0.1382 | True               |
| youtube           | twitter           | ML             | randomforest       | randomforest__fasttext    | manual          |           0.0971 |     0.0808 |     0.1143 | True               |
| youtube           | twitter           | Sequence       | bilstm             | bilstm__random            | original        |           0.1544 |     0.1364 |     0.1731 | True               |
| youtube           | twitter           | Transformer    | arabertv2          | arabertv2                 | original        |          -0.0033 |    -0.0138 |     0.0071 | False              |

**Interpretation.** Cross-platform performance is consistently lower than in-domain performance at the family level, and the family-wise Wilcoxon tests are significant in both transfer directions for ML, sequence, and transformer models. The best Twitter-to-YouTube transfer result is the transformer under manual preprocessing (Macro-F1=0.8364), while the best YouTube-to-Twitter transfer result is again a transformer under the original view (Macro-F1=0.9433). Transformers also have the smallest mean transfer loss in both directions. One nuance remains important: for the strongest YouTube-to-Twitter transformer configuration, the bootstrap delta CI [-0.0138, 0.0071] crosses zero, which means that a very strong transformer can occasionally transfer almost as well as, or slightly better than, its in-domain score.

## Final Decision Summary

| hypothesis   | support_verdict     | null_hypothesis_language                                                                                              | interpretation                                                                                                                                                                   |
|:-------------|:--------------------|:----------------------------------------------------------------------------------------------------------------------|:---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------|
| H1           | Supported           | Reject the null of no transformer-family advantage on both platforms.                                                 | Transformer family comparisons were significant on both platforms; 2 of 6 transformer-only view summaries also showed a significant lead for the top backbone after BH-FDR.      |
| H2           | Partially supported | Reject the null only for selected preprocessing contrasts, not uniformly across all families.                         | Family-wise Wilcoxon tests were significant in 3 of 12 contrasts, and best-config McNemar tests were significant in 4 of 12 reported contrasts after BH-FDR.                     |
| H3           | Supported           | Reject the null of no transfer loss at the family level; transformers show the smallest mean loss in both directions. | Family-wise transfer-loss tests were significant in all reported family comparisons, and the transformer family had the smallest mean Macro-F1 loss in both transfer directions. |

### Practical Reading of the Results

- The in-domain story is clear: transformers are the strongest family on both platforms.
- The preprocessing story is selective: `manual` cleaning is often harmful, while `deepfake_aware` is usually less harmful but not consistently beneficial.
- The error-analysis story is semantic: omission and contradiction remain the dominant hard cases.
- The transfer story is cautious: cross-platform robustness remains limited, although transformers are the most stable family overall.